# RFM Segmentation Analysis
**Re-Emulator | Vintage Sampling Media Resale | Jan 2025 – Jun 2026**

RFM (Recency, Frequency, Monetary) segmentation is a proven framework for understanding customer behavior and identifying actionable customer segments.

Each customer is scored on three dimensions:
- **Recency (R):** How recently did they purchase? More recent = higher score.
- **Frequency (F):** How many times have they purchased? More purchases = higher score.
- **Monetary (M):** How much have they spent in total? Higher spend = higher score.

Customers are then grouped into segments that drive targeted business decisions.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Load cleaned order-level data
df = pd.read_csv('../data/orders_clean.csv', parse_dates=['order_date'])

snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

print(f'Orders: {len(df):,}')
print(f'Unique buyers: {df["buyer_id"].nunique():,}')
print(f'Snapshot date: {snapshot_date.date()}')

## 2. Build RFM Features

In [ ]:
# Compute raw RFM metrics per customer
rfm = df.groupby('buyer_id').agg(
    recency   = ('order_date',  lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_date',  'count'),
    monetary  = ('order_value', 'sum'),
).reset_index()

print('RFM summary:')
print(rfm[['recency','frequency','monetary']].describe())

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(rfm['recency'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Recency Distribution\n(days since last purchase)')
axes[0].set_xlabel('Days')

axes[1].hist(rfm['frequency'].clip(upper=rfm['frequency'].quantile(0.95)),
             bins=30, color='steelblue', edgecolor='white')
axes[1].set_title('Frequency Distribution\n(95th pct cap)')
axes[1].set_xlabel('Orders')

axes[2].hist(rfm['monetary'].clip(upper=rfm['monetary'].quantile(0.95)),
             bins=30, color='steelblue', edgecolor='white')
axes[2].set_title('Monetary Distribution\n(95th pct cap)')
axes[2].set_xlabel('Total Spend ($)')

plt.tight_layout()
plt.savefig('../outputs/rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. RFM Scoring

Each dimension is scored 1-5 using quintiles.

**Note on Recency scoring:** Lower recency (purchased more recently) = higher score. The scoring is inverted for recency.

In [ ]:
# Score each dimension 1-5 using quintiles
rfm['R_score'] = pd.qcut(rfm['recency'],  q=5, labels=[5,4,3,2,1]).astype(int)  # inverted
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)

# Composite RFM score
rfm['RFM_score'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']
rfm['RFM_segment_code'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

print('RFM score summary:')
print(rfm[['R_score','F_score','M_score','RFM_score']].describe())

## 4. Customer Segmentation

Customers are mapped to actionable segments based on their RFM scores.

In [ ]:
def assign_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    score = row['RFM_score']
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f <= 2:
        return 'Promising'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f >= 4:
        return 'Cant Lose Them'
    elif r <= 2 and score <= 6:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

seg_counts = rfm['segment'].value_counts()
print('Customers per segment:')
print(seg_counts)

In [ ]:
# Segment summary stats
seg_summary = rfm.groupby('segment').agg(
    n_customers   = ('buyer_id',   'count'),
    avg_recency   = ('recency',    'mean'),
    avg_frequency = ('frequency',  'mean'),
    avg_monetary  = ('monetary',   'mean'),
    total_revenue = ('monetary',   'sum'),
).round(2).sort_values('total_revenue', ascending=False)

print('\nSegment summary:')
print(seg_summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Customer count by segment
seg_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Customers by Segment')
axes[0].set_xlabel('')
axes[0].set_ylabel('Customers')
axes[0].tick_params(axis='x', rotation=35)

# Revenue by segment
seg_summary['total_revenue'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Total Revenue by Segment')
axes[1].set_xlabel('')
axes[1].set_ylabel('Revenue ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig('../outputs/rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. K-Means Clustering on RFM Features

Rule-based segmentation works well but k-means clustering lets the data determine natural groupings without predefined rules.

We standardize RFM features first since they are on different scales, then use the elbow method to select the optimal number of clusters.

In [ ]:
# Standardize RFM features
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency','frequency','monetary']])

# Elbow method — find optimal k
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(rfm_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('../outputs/rfm_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range[silhouettes.index(max(silhouettes))]
print(f'\nOptimal k by silhouette score: {best_k}')

In [ ]:
# Fit final k-means model
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm['cluster'] = km_final.fit_predict(rfm_scaled)

# Cluster profiles
cluster_profile = rfm.groupby('cluster').agg(
    n_customers   = ('buyer_id',   'count'),
    avg_recency   = ('recency',    'mean'),
    avg_frequency = ('frequency',  'mean'),
    avg_monetary  = ('monetary',   'mean'),
    total_revenue = ('monetary',   'sum'),
).round(2)

print('K-means cluster profiles:')
print(cluster_profile)

In [ ]:
# Scatter plot — Frequency vs Monetary colored by cluster
plt.figure(figsize=(8, 5))
colors = ['steelblue', 'coral']
for cluster_id in rfm['cluster'].unique():
    mask = rfm['cluster'] == cluster_id
    plt.scatter(
        rfm.loc[mask, 'frequency'].clip(upper=rfm['frequency'].quantile(0.95)),
        rfm.loc[mask, 'monetary'].clip(upper=rfm['monetary'].quantile(0.95)),
        c=colors[cluster_id],
        label=f'Cluster {cluster_id}',
        alpha=0.6,
        edgecolors='none'
    )
plt.legend()
plt.title('K-Means Clusters: Frequency vs Monetary Value')
plt.xlabel('Purchase Frequency (orders)')
plt.ylabel('Total Spend ($)')
plt.tight_layout()
plt.savefig('../outputs/rfm_clusters_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Business Implications

RFM segments translate directly into marketing and retention actions:

| Segment | Action |
|---|---|
| **Champion** | Reward loyalty, ask for reviews, early access to new inventory |
| **Loyal** | Upsell higher value items, maintain engagement |
| **At Risk** | Win-back campaign, personalized outreach |
| **Cant Lose Them** | Aggressive re-engagement, special offers |
| **New Customer** | Onboarding, introduce to product range |
| **Lost** | Low investment, broad reactivation only |

In a niche resale context like vintage sampling media, Champions and Loyal customers are especially valuable — they understand the product, buy repeatedly, and are unlikely to be price sensitive on rare items.

In [ ]:
# Save enriched RFM table
rfm.to_csv('../data/rfm_output.csv', index=False)
print('Saved: rfm_output.csv')